In [1]:
!pip install transformers datasets sentencepiece accelerate rouge-score -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.7/47.7 MB 42.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 93.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 72.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 49.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 8.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 33.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 14.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 9.9 MB/s eta 0:00:000:00:0100:01
ERROR: pip's d

In [2]:
!gdown --id 1heHm5FaSaULfOREzG3mSLt7wAxaO-9wx

/usr/local/lib/python3.11/dist-packages/gdown/__main__.py:140: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From: https://drive.google.com/uc?id=1heHm5FaSaULfOREzG3mSLt7wAxaO-9wx
To: /kaggle/working/final_data.json
100%|██████████████████████████████████████| 15.4M/15.4M [00:00<00:00, 72.4MB/s]


In [3]:
import os
import re
import hashlib
import random
import json
import numpy as np
import pandas as pd
import torch
from tqdm import tqdm
from torch.utils.data import Dataset
from sklearn.model_selection import GroupShuffleSplit
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from rouge_score import rouge_scorer

# ==========================================
# 2. CONFIGURATION
# ==========================================
CFG = {
    "data_path": "/kaggle/working/final_data.json",
    "output_dir": "./controlable-question-generation-GPT2",
    "base_model": "gpt2",
    "epochs": 3,            # reasonable for Kaggle / full FT
    "batch_size": 1,        # per-device batch size
    "grad_accum": 4,
    "lr": 5e-5,
    "max_len": 512,         # total sequence length (source + target)
    "max_passage_words": 250,  # cap passage length in words
    "min_target_tokens": 10,   # minimum target length to keep example
    "seed": 42,
}

IRT_PROXY_MAP = {"Easy": -1.0, "Medium": 0.0, "Hard": 1.0}

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG["seed"])

# ==========================================
# 3. DATA PROCESSING
# ==========================================
def load_data(path):
    print(f"📂 Loading data from {path}...")
    try:
        df = pd.read_json(path)
        if "data" in df.columns:
            df = pd.DataFrame(df["data"].tolist())
    except ValueError:
        df = pd.read_json(path, lines=True)

    # Rename columns to standard format
    rename_map = {
        "PassageContext": "passage",
        "QuestionType": "question_type",
        "InstructionContext": "ref_instruction",
        "Question": "ref_question",
        "Answer": "ref_answer",
    }
    df = df.rename(columns=rename_map)
    
    # Filter valid rows
    df = df.dropna(subset=["passage", "ref_question"])
    df = df[df["passage"].str.len() > 50]
    
    # Create Split ID
    if "passage_id" not in df.columns:
        df["passage_id"] = df["passage"].apply(
            lambda x: hashlib.md5(str(x).encode()).hexdigest()
        )
    
    return df

def normalize_text(text):
    return (
        str(text)
        .replace("SEP", "<SEP>")
        .replace("BLANK", "<BLANK>")
        .strip()
    )

def _canon_type(s: str) -> str:
    return str(s).upper().strip().replace("-", "_").replace(" ", "_")

# Natural language prefix for each question type
TYPE_PREFIX_MAP = {
    "READINGTRUEFALSENOTGIVEN": "Generate a TRUE/FALSE/NOT GIVEN reading question.",
    "READINGYESNONOTGIVEN": "Generate a YES/NO/NOT GIVEN reading question.",
    "READINGMULTIPLECHOICES": "Generate a multiple-choice reading question.",
    "READINGSHORTANSWER": "Generate a short-answer reading question.",
    "READINGMATCHINGHEADINGS": "Generate a matching headings reading question.",
    "READINGMATCHINGFEATURES": "Generate a matching features reading question.",
    "READINGMATCHINGENDINGS": "Generate a matching sentence endings reading question.",
    "READINGTEXTCOMPLETION": "Generate a text completion reading question.",
    "READINGSELECTIVETEXTCOMPLETION": "Generate a selective text completion reading question.",
    "READINGTABLECOMPLETION": "Generate a table completion reading question.",
}

def get_irt_prefix(row):
    diff = str(row.get("difficulty", "Medium")).strip().title()
    qtype_raw = str(row.get("question_type", "")).strip()
    qtype_canon = _canon_type(qtype_raw)
    b_val = IRT_PROXY_MAP.get(diff, 0.0)

    type_sentence = TYPE_PREFIX_MAP.get(
        qtype_canon,
        "Generate an IELTS reading question.",
    )
    return f"[a=1.0, b={b_val}, c=0.0] {type_sentence} <TYPE_{qtype_canon}>"

# ==========================================
# 4. DATASET WITH SAFE LABEL MASKING
# ==========================================
class GPT2ConditionalDataset(Dataset):
    """
    full_text = source + target
      source = IRT + TYPE + Passage + <SEP>
      target = Instruction/Question/Answer + <END>
    labels:
      source tokens -> -100 (ignored)
      target tokens -> supervised
    """

    def __init__(self, df, tokenizer, max_len, max_passage_words, min_target_tokens):
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.max_passage_words = max_passage_words
        self.min_target_tokens = min_target_tokens
        self.data = []

        kept = 0
        skipped_short_target = 0

        for _, row in df.iterrows():
            prefix = get_irt_prefix(row)
            passage = normalize_text(row["passage"])

            # Truncate passage by words to avoid extremely long source
            passage_words = passage.split()
            if len(passage_words) > self.max_passage_words:
                passage = " ".join(passage_words[:self.max_passage_words])

            ins = normalize_text(row.get("ref_instruction", ""))
            ques = normalize_text(row.get("ref_question", ""))
            ans  = normalize_text(row.get("ref_answer", ""))

            if not ques:
                continue

            source_txt = f"{prefix}\nPassage: {passage} <SEP>"
            target_txt = f"\nInstruction: {ins}\nQuestion: {ques}\nAnswer: {ans} <END>"

            # Check target token length; skip degenerate very-short targets
            tgt_ids = self.tokenizer(
                target_txt, add_special_tokens=False
            )["input_ids"]
            if len(tgt_ids) < self.min_target_tokens:
                skipped_short_target += 1
                continue

            self.data.append((source_txt, target_txt))
            kept += 1

        print(f"✅ Built {kept} examples (skipped {skipped_short_target} with very short targets)")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        source_txt, target_txt = self.data[idx]

        src_ids = self.tokenizer(
            source_txt, add_special_tokens=False
        )["input_ids"]
        tgt_ids = self.tokenizer(
            target_txt, add_special_tokens=False
        )["input_ids"]

        max_src_len = int(self.max_len * 0.5)
        max_tgt_len = self.max_len - max_src_len

        if len(src_ids) > max_src_len:
            src_ids = src_ids[:max_src_len]

        if len(tgt_ids) > max_tgt_len:
            tgt_ids = tgt_ids[:max_tgt_len]

        full_ids = src_ids + tgt_ids
        src_len = len(src_ids)

        # labels: mask source tokens
        labels = [-100] * src_len + full_ids[src_len:]

        # Padding
        pad_len = self.max_len - len(full_ids)
        if pad_len > 0:
            full_ids = full_ids + [self.tokenizer.pad_token_id] * pad_len
            labels   = labels   + [-100] * pad_len
            attention_mask = [1] * (len(full_ids) - pad_len) + [0] * pad_len
        else:
            attention_mask = [1] * self.max_len

        return {
            "input_ids": torch.tensor(full_ids, dtype=torch.long),
            "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
            "labels": torch.tensor(labels, dtype=torch.long),
        }

# ==========================================
# 5. MODEL & TOKENIZER SETUP
# ==========================================
df = load_data(CFG["data_path"])

# Split by passage_id
splitter = GroupShuffleSplit(n_splits=1, test_size=0.1, random_state=CFG["seed"])
train_idx, val_idx = next(splitter.split(df, groups=df["passage_id"]))
train_df, val_df = df.iloc[train_idx], df.iloc[val_idx]

print(f"Train size: {len(train_df)}, Val size: {len(val_df)}")

# Tokenizer
tokenizer = GPT2TokenizerFast.from_pretrained(CFG["base_model"])

# Build type tokens
type_tokens = [
    f"<TYPE_{_canon_type(t)}>"
    for t in df["question_type"].dropna().unique()
]

special_tokens = {
    "pad_token": "<PAD>",
    "eos_token": "<END>",
    "additional_special_tokens": ["<SEP>", "<BLANK>"] + list(sorted(set(type_tokens))),
}

tokenizer.add_special_tokens(special_tokens)
tokenizer.pad_token = "<PAD>"
tokenizer.eos_token = "<END>"

print("Special tokens:", special_tokens)

# Model
model = GPT2LMHeadModel.from_pretrained(CFG["base_model"])
model.resize_token_embeddings(len(tokenizer))

# Memory helpers (for Kaggle)
model.config.use_cache = False
if hasattr(model, "gradient_checkpointing_enable"):
    model.gradient_checkpointing_enable()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
print("Using device:", device)

# Datasets
train_ds = GPT2ConditionalDataset(
    train_df,
    tokenizer,
    max_len=CFG["max_len"],
    max_passage_words=CFG["max_passage_words"],
    min_target_tokens=CFG["min_target_tokens"],
)
val_ds   = GPT2ConditionalDataset(
    val_df,
    tokenizer,
    max_len=CFG["max_len"],
    max_passage_words=CFG["max_passage_words"],
    min_target_tokens=CFG["min_target_tokens"],
)

print("\n🔍 DEBUG: Inspecting one training sample...")
sample = train_ds[0]
input_ids = sample["input_ids"]
labels    = sample["labels"]

print(f"Input IDs shape: {input_ids.shape}")
print(f"Labels shape: {labels.shape}")

decoded_input = tokenizer.decode(input_ids, skip_special_tokens=False)
print(f"\n▶️ DECODED INPUT (Truncated):\n{decoded_input[:400]}...")

masked_count = (labels == -100).sum().item()
total_count  = labels.numel()
print(f"\nTarget tokens (visible to loss): {total_count - masked_count}")
print(f"Masked tokens (ignored): {masked_count}")

if masked_count == 0:
    print("❌ WARNING: No masking detected! Model will fail.")
elif masked_count == total_count:
    print("❌ WARNING: All masked! Model will learn nothing.")
else:
    print("✅ Masking looks correct. The model will only learn to generate the Target.")
# -----------------------------------

# ==========================================
# 6. TRAINING SETUP
# ==========================================
args = TrainingArguments(
    output_dir=CFG["output_dir"],
    per_device_train_batch_size=CFG["batch_size"],
    gradient_accumulation_steps=CFG["grad_accum"],
    num_train_epochs=CFG["epochs"],
    learning_rate=CFG["lr"],
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    save_strategy="epoch",
    eval_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    prediction_loss_only=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=default_data_collator,
)

print("\n🚀 Starting Full Fine-Tuning...")
trainer.train()

# Save
trainer.save_model(CFG["output_dir"])
tokenizer.save_pretrained(CFG["output_dir"])
print("✅ Model Saved at", CFG["output_dir"])


2025-12-12 16:10:59.334243: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765555859.508138      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765555859.560044      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

Using device: cuda
✅ Built 2305 examples (skipped 0 with very short targets)
✅ Built 291 examples (skipped 0 with very short targets)

🔍 DEBUG: Inspecting one training sample...
Input IDs shape: torch.Size([512])
Labels shape: torch.Size([512])

▶️ DECODED INPUT (Truncated):
[a=1.0, b=-1.0, c=0.0] Generate a TRUE/FALSE/NOT GIVEN reading question. <TYPE_READINGTRUEFALSENOTGIVEN>
Passage: During the sixth and seventh centuries, the inhabitants of the modern-day states of Gujarat and Rajasthan in North-western India developed a method of gaining access to clean, fresh groundwater during the dry season for drinking, bathing, watering animals and irrigation. However, the s...

Target tokens (visible to loss): 84
Masked tokens (ignored): 428
✅ Masking looks correct. The model will only learn to generate the Target.

🚀 Starting Full Fine-Tuning...


`loss_type=None` was set in the config but it is unrecognised.Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,0.696000,1.135230
2,0.489000,1.094646
3,0.444400,1.077151


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


✅ Model Saved at ./controlable-question-generation-GPT2


In [6]:
# ==========================================
# 7. EVALUATION
# ==========================================
print("🧪 Starting Evaluation...")
model.eval()
model.to(device)
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
scores = []
results = []

# Evaluate on first 200 validation samples
test_samples = val_df.head(200)

for i, row in tqdm(test_samples.iterrows(), total=len(test_samples)):
    try:
        prefix = get_irt_prefix(row)
        passage = normalize_text(row["passage"])

        # Truncate passage for prompt as well (same policy)
        passage_words = passage.split()
        if len(passage_words) > CFG["max_passage_words"]:
            passage = " ".join(passage_words[:CFG["max_passage_words"]])

        prompt = f"{prefix}\nPassage: {passage} <SEP>"

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=CFG["max_len"],
        ).to(device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,   # deterministic (good for evaluation)
                num_beams=4,       # beam search for more stable outputs
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.convert_tokens_to_ids("<END>"),
                repetition_penalty=1.1,
            )

        decoded = tokenizer.decode(outputs[0], skip_special_tokens=False)

        # Extract generated part after <SEP>
        if "<SEP>" in decoded:
            generated = decoded.split("<SEP>", 1)[-1]
        else:
            generated = decoded[len(prompt):]

        if generated is None:
            generated = ""

        generated = generated.replace("<END>", "").strip()

        # ---- Parse generated Instruction / Question / Answer ----
        gen_instruction = ""
        gen_question = ""
        gen_answer = ""

        lines = [l.strip() for l in generated.splitlines() if l.strip()]
        for line in lines:
            if line.startswith("Instruction:"):
                gen_instruction = line[len("Instruction:"):].strip()
            elif line.startswith("Question:"):
                gen_question = line[len("Question:"):].strip()
            elif line.startswith("Answer:"):
                gen_answer = line[len("Answer:"):].strip()

        # ---- Build reference text for ROUGE ----
        ref_instruction = row.get("ref_instruction", "")
        ref_question    = row.get("ref_question", "")
        ref_answer      = row.get("ref_answer", "")

        ref = (
            f"Instruction: {ref_instruction} "
            f"Question: {ref_question} "
            f"Answer: {ref_answer}"
        )

        try:
            score = scorer.score(ref, generated)["rougeL"].fmeasure
        except Exception as e:
            print("ROUGE error:", e)
            score = 0.0

        scores.append(score)

        # ---- Add all requested columns to results ----
        results.append({
            # old fields (keep if you still want them)
            "ref": ref,
            "gen": generated,
            "score": score,

            # reference columns
            "reference_passage":       row.get("passage", ""),
            "reference_difficulty":    row.get("difficulty", ""),
            "reference_category":      row.get("category", "") or row.get("Category", ""),
            "reference_question_type": row.get("question_type", "") or row.get("QuestionType", ""),
            "reference_instruction":   ref_instruction,
            "reference_question":      ref_question,
            "reference_answer":        ref_answer,

            # generated columns
            "generated_full_output": generated,
            "generated_instruction":  gen_instruction,
            "generated_question":     gen_question,
            "generated_answer":       gen_answer,
        })

    except Exception as e:
        print("Error in evaluation loop:", e)

df_res = pd.DataFrame(results)
print("\nSample evaluation rows:")
print(df_res.head())

print(f"\n📊 Final Average ROUGE-L: {np.mean(scores):.4f}")
df_res.to_csv("generated_samples_gpt2.csv", index=False)
print("💾 Saved evaluation results to generated_samples_gpt2.csv")

🧪 Starting Evaluation...


100%|██████████| 200/200 [03:24<00:00,  1.02s/it]


Sample evaluation rows:
                                                 ref  \
0  Instruction: Do the following statements agree...   
1  Instruction: Do the following statements agree...   
2  Instruction: Do the following statements agree...   
3  Instruction: Do the following statements agree...   
4  Instruction: Do the following statements agree...   

                                                 gen     score  \
0  Instruction: Do the following statement agree ...  0.360360   
1  Instruction: Do the following statement agree ...  0.355140   
2  Instruction: Do the following statement agree ...  0.380952   
3  Instruction: Do the following statement agree ...  0.355140   
4  Instruction: Do the following statement agree ...  0.355140   

                                   reference_passage reference_difficulty  \
0  [A] Alan Macfarlane, professor of anthropologi...                 Hard   
1  [A] Alan Macfarlane, professor of anthropologi...                 Easy   
2  [A] Ala

In [7]:
result = pd.read_csv("/kaggle/working/generated_samples_gpt2.csv")

In [8]:
result.shape

(200, 14)

In [9]:
result.head()

,ref,gen,score,reference_passage,reference_difficulty,reference_category,reference_question_type,reference_instruction,reference_question,reference_answer,generated_full_output,generated_instruction,generated_question,generated_answer
0,Instruction: Do the following statements agree...,Instruction: Do the following statement agree ...,0.360360,"[A] Alan Macfarlane, professor of anthropologi...",Hard,History and Culture,readingTrueFalseNotGiven,Do the following statements agree with the inf...,China’s transport system was not suitable for ...,NOT GIVEN,Instruction: Do the following statement agree ...,Do the following statement agree with the info...,"[A] Alan Macfarlane, professor of anthropologi...",TRUE
1,Instruction: Do the following statements agree...,Instruction: Do the following statement agree ...,0.355140,"[A] Alan Macfarlane, professor of anthropologi...",Easy,History and Culture,readingTrueFalseNotGiven,Do the following statements agree with the inf...,Tea and beer both helped to prevent dysentery ...,TRUE,Instruction: Do the following statement agree ...,Do the following statement agree with the info...,"[A] Alan Macfarlane, professor of anthropologi...",TRUE
2,Instruction: Do the following statements agree...,Instruction: Do the following statement agree ...,0.380952,"[A] Alan Macfarlane, professor of anthropologi...",Easy,History and Culture,readingTrueFalseNotGiven,Do the following statements agree with the inf...,Roy Porter disagrees with Professor Macfarlane...,FALSE,Instruction: Do the following statement agree ...,Do the following statement agree with the info...,"[A] Alan Macfarlane, professor of anthropologi...",TRUE
3,Instruction: Do the following statements agree...,Instruction: Do the following statement agree ...,0.355140,"[A] Alan Macfarlane, professor of anthropologi...",Easy,History and Culture,readingTrueFalseNotGiven,Do the following statements agree with the inf...,After 1740，there was a reduction in population...,FALSE,Instruction: Do the following statement agree ...,Do the following statement agree with the info...,"[A] Alan Macfarlane, professor of anthropologi...",TRUE
4,Instruction: Do the following statements agree...,Instruction: Do the following statement agree ...,0.355140,"[A] Alan Macfarlane, professor of anthropologi...",Hard,History and Culture,readingTrueFalseNotGiven,Do the following statements agree with the inf...,People in Britain used to make beer at home.,NOT GIVEN,Instruction: Do the following statement agree ...,Do the following statement agree with the info...,"[A] Alan Macfarlane, professor of anthropologi...",TRUE


In [10]:
print("REF:\n", result.iloc[0]['ref'])
print("GEN: \n", result.iloc[0]['gen'])

REF:
 Instruction: Do the following statements agree with the information given in Reading Passage 1?
In boxes 8-13 on your answer sheet, write
TRUE if the statement agrees with the information
FALSE if the statement contradicts the information
NOT GIVEN if there is no information on this Question: China’s transport system was not suitable for industry in the 18th century. Answer: NOT GIVEN
GEN: 
 Instruction: Do the following statement agree with the information in the passage?
Write TRUE, FALSE or NOT GIVEN.
Question: [A] Alan Macfarlane, professor of anthropological science at King’s College, Cambridge has, like other historians, spent decades wrestling with the enigma of the Industrial Revolution.
Answer: TRUE


In [ ]:
from huggingface_hub import login

login(token="")

In [12]:
HUB_MODEL_ID = "alikarimiaca/controlable-question-generation-GPT2"
model.push_to_hub(HUB_MODEL_ID)
tokenizer.push_to_hub(HUB_MODEL_ID)
trainer.push_to_hub(HUB_MODEL_ID)
print("\nAll done! Check your Hugging Face profile.")

README.md: 0.00B [00:00, ?B/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

No files have been modified since last commit. Skipping to prevent empty commit.


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


All done! Check your Hugging Face profile.
